# 19. Scaled MetaQA with an LLM-Extracted Graph: Is the Gain the Graph or the Curation?

**Presentation map:** Slide 32 (vector RAG is the baseline), Slides 36 and 38 (graphs help on multi-hop reasoning),
Slides 40-41 (traversal lifts multi-hop KGQA).

Notebook 8 found gains of 40-60 points for a graph agent on MetaQA, but its graph is the **curated** WikiMovies
knowledge base, whose facts are exactly the benchmark's answers. Notebook 11 used an **LLM-extracted** graph and
found gains of 5-12 F1 points, but on different benchmarks (MuSiQue, 2WikiMultiHopQA). This notebook keeps the
benchmark fixed and changes only where the graph comes from.

**Data.** MetaQA *vanilla* test questions (`data/benchmarks/metaqa/`, as in notebook 8) and the WikiMovies text
that MetaQA's knowledge base was built from: **18,127 Wikipedia movie articles** (first paragraph,
`wiki.txt` from the MovieQA release of Miller et al. 2016, `data/benchmarks/metaqa/wikimovies_text/`, CC BY 3.0).
We sample **300 questions per hop** (seed 7) **whose topic entity is a movie with an article**. Questions whose topic
is a person, year, genre or tag have no article to start from and are excluded, so these samples differ from
notebook 8's.

**Systems** (all `gpt-6-luna`, no reasoning, `smash_batch.py`; extraction and vector RAG via the Batch API, the agents
with synchronous calls):
- **(a) Vector RAG over the articles.** One chunk per article (title + text), `text-embedding-3-small`, top 10.
- **(b) Graph agent over the curated KB.** Notebook 8's agent and its single read-only tool
  `get_neighbors(entities, relation)`, unchanged except for the model.
- **(c) The same agent over an LLM-extracted graph.** The LLM reads each article and extracts facts about the
  film in the KB's relation vocabulary (`directed_by`, `written_by`, `starred_actors`, `release_year`,
  `in_language`, `has_genre`, `has_tags`, `has_imdb_rating`, `has_imdb_votes`). Entities are linked by normalised
  name (case, accents and punctuation ignored). All 18,127 articles are extracted, so the graph does not depend on
  the sampled questions.

**Scoring:** Hits@1 and F1 against MetaQA's gold answer sets, string matching as in notebook 8, with names compared
after the same normalisation for all three systems. Cached in `data/benchmarks/metaqa/cache_19/`.

In [1]:
import json, os, random, re, unicodedata
from collections import Counter, defaultdict
from typing import Literal
import numpy as np
import networkx as nx
from pydantic import BaseModel
from smash_batch import LLM, mode_from_env, wilson, paired_boot

MODE = mode_from_env()
TEST = MODE == "test"
DATA = "data/benchmarks/metaqa"
llm = LLM(f"{DATA}/cache_19" + ("_test" if TEST else ""))
SEED, N_PER_HOP = 7, (10 if TEST else 300)
RELS = ["directed_by", "written_by", "starred_actors", "release_year", "in_language", "has_genre", "has_tags",
        "has_imdb_rating", "has_imdb_votes"]

def key(s):
    # Entity linking and scoring key: lower case, accents and punctuation removed, whitespace collapsed.
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", s)).strip()

# --- Curated KB (notebook 8) ---
KB = nx.MultiDiGraph()
for line in open(f"{DATA}/kb.txt", encoding="utf-8"):
    h, r, t = line.rstrip("\n").split("|")
    KB.add_edge(h, t, key=r)
movies_kb = {h for h, _, _ in KB.edges(keys=True)}

# --- WikiMovies articles: '1 <title>' starts an article, numbered lines follow, a blank line ends it ---
articles, cur = [], None
for line in open(f"{DATA}/wikimovies_text/movieqa/knowledge_source/wiki.txt", encoding="utf-8"):
    line = line.rstrip("\n")
    if not line:
        cur = None; continue
    n, txt = line.split(" ", 1)
    if n == "1":
        cur = {"title": txt, "lines": []}; articles.append(cur)
    elif cur is not None:
        cur["lines"].append(txt)
strip_title = lambda t: re.sub(r"\s*\((?:[^()]*\bfilm|[^()]*\bmovie)\)$", "", t)
for i, a in enumerate(articles):
    a["id"], a["movie"], a["text"] = f"a{i:05d}", strip_title(a["title"]), " ".join(a["lines"])
movie_keys = {key(a["movie"]) for a in articles}
print(f"{len(articles):,} articles, {np.mean([len(a['text']) for a in articles]):.0f} characters on average; "
      f"{sum(key(m) in movie_keys for m in movies_kb):,} of {len(movies_kb):,} KB movies have an article")

# --- Questions: topic entity must be a KB movie with an article ---
def load_split(hop):
    rows = []
    for i, line in enumerate(open(f"{DATA}/qa_test_{hop}hop.txt", encoding="utf-8")):
        q, a = line.rstrip("\n").split("\t")
        rows.append({"qid": f"{hop}hop-{i}", "hop": hop, "question": q,
                     "topic": re.search(r"\[(.+?)\]", q).group(1), "gold": a.split("|")})
    return rows
splits = {h: load_split(h) for h in (1, 2, 3)}
eligible = {h: [q for q in s if q["topic"] in movies_kb and key(q["topic"]) in movie_keys] for h, s in splits.items()}
rng = random.Random(SEED)
Q = [q for h in (1, 2, 3) for q in rng.sample(eligible[h], N_PER_HOP)]
print("Eligible test questions (topic is a movie with an article): " +
      ", ".join(f"{h}-hop {len(eligible[h]):,} of {len(splits[h]):,}" for h in (1, 2, 3)))
print(f"Sampled {len(Q)} questions ({N_PER_HOP} per hop, seed {SEED})")

18,127 articles, 545 characters on average; 16,425 of 16,427 KB movies have an article
Eligible test questions (topic is a movie with an article): 1-hop 7,201 of 9,947, 2-hop 3,167 of 14,872, 3-hop 14,271 of 14,274
Sampled 900 questions (300 per hop, seed 7)


## Extracting a graph from the articles

One extraction call per article. The subject of every fact is the article's film; the relations are the KB's nine.
Genres are restricted to the KB's 23 genre names (a closed vocabulary such as any ontology would define); all other
values are free text. Facts the text does not state must not be guessed.

In [2]:
GENRES = sorted({t for _, t, r in KB.edges(keys=True) if r == "has_genre"})
class Fact(BaseModel):
    relation: Literal[tuple(RELS)]
    value: str
class Facts(BaseModel):
    facts: list[Fact]
EXTRACT_SYS = (
    "You build a movie knowledge graph from Wikipedia text. Extract facts about the film the article is about, "
    "using only these relations (the film is always the subject):\n"
    "- directed_by: a director (one person per fact)\n- written_by: a screenwriter or author of the source work\n"
    "- starred_actors: an actor or actress who stars in the film\n- release_year: the four-digit release year\n"
    "- in_language: a language of the film (e.g. English, French)\n"
    f"- has_genre: one of {', '.join(GENRES)}\n"
    "- has_tags: a short lower-case keyword describing the film (topic, setting, source, notable feature)\n"
    "- has_imdb_rating / has_imdb_votes: only if the text states the IMDb rating or popularity\n"
    "One value per fact; use full names as written in the text. Extract only what the text states; do not guess.")
EX_IDS = ([a["id"] for a in articles] if not TEST else
          sorted({a["id"] for a in articles if key(a["movie"]) in {key(q["topic"]) for q in Q}}))
by_id = {a["id"]: a for a in articles}
ext = llm.run("extract", {i: LLM.body([{"role": "system", "content": EXTRACT_SYS},
                                       {"role": "user", "content": f"Article: {by_id[i]['title']}\n\n{by_id[i]['text']}"}],
                                      max_tokens=800, schema=Facts) for i in EX_IDS}, parse=Facts)
bad = sum(r["parsed"] is None for r in ext.values())

# Entity linking by normalised name: every surface form with the same key becomes one node, named by its most
# frequent surface form.
surface, triples = defaultdict(Counter), set()
for i in EX_IDS:
    m = by_id[i]["movie"]
    surface[key(m)][m] += 1
    for f in (ext[i]["parsed"] or {"facts": []})["facts"]:
        v = f["value"].strip()
        if f["relation"] == "release_year":
            y = re.search(r"\b(1[89]\d\d|20\d\d)\b", v)
            if not y:
                continue
            v = y.group(1)
        if not key(v):
            continue
        surface[key(v)][v] += 1
        triples.add((key(m), f["relation"], key(v)))
name = {k: c.most_common(1)[0][0] for k, c in surface.items()}
EX = nx.MultiDiGraph()
for h, r, t in triples:
    EX.add_edge(name[h], name[t], key=r)
print(f"Extracted graph: {EX.number_of_nodes():,} entities, {EX.number_of_edges():,} triples from {len(EX_IDS):,} "
      f"articles ({bad} outputs did not parse)")
print("  " + ", ".join(f"{r} {n:,}" for r, n in Counter(r for _, _, r in EX.edges(keys=True)).most_common()))
print(f"Curated KB:      {KB.number_of_nodes():,} entities, {KB.number_of_edges():,} triples")

Extracted graph: 52,117 entities, 146,006 triples from 18,127 articles (0 outputs did not parse)
  starred_actors 49,295, has_tags 19,839, has_genre 19,179, release_year 17,272, written_by 16,979, directed_by 16,802, in_language 6,635, has_imdb_rating 5
Curated KB:      43,234 entities, 133,582 triples


### How much of the curated KB does extraction recover?

Triple recall: the share of curated KB triples about a film that also appear in the extracted graph (after name
normalisation), per relation, for the topic films of the sampled questions and for all films with an article.

In [3]:
ex_set = {(key(h), r, key(t)) for h, t, r in EX.edges(keys=True)}
def triple_recall(films):
    fk = {key(f) for f in films}
    out = {}
    for r in RELS + ["ALL"]:
        T = {(key(h), rr, key(t)) for h, t, rr in KB.edges(keys=True) if key(h) in fk and (r == "ALL" or rr == r)}
        if T:
            out[r] = (len(T & ex_set) / len(T), len(T))
    return out
REC = {"topic films": triple_recall({q["topic"] for q in Q}),
       "all films": triple_recall({m for m in movies_kb if key(m) in movie_keys} if not TEST else {q["topic"] for q in Q})}
print(f"{'relation':<18}" + "".join(f"{c:>26}" for c in REC))
for r in RELS + ["ALL"]:
    print(f"{r:<18}" + "".join(f"{REC[c][r][0]:>17.0%} of {REC[c][r][1]:>6,}" if r in REC[c] else f"{'-':>26}" for c in REC))

relation                         topic films                 all films
directed_by                     97% of    933              98% of 15,955
written_by                      78% of  1,181              77% of 19,122
starred_actors                  96% of  2,046              96% of 33,714
release_year                    99% of    946              98% of 16,712
in_language                     62% of    200              68% of  3,428
has_genre                       90% of    874              90% of 15,477
has_tags                         7% of  1,868               9% of 28,713
has_imdb_rating                  0% of     14               0% of    314
has_imdb_votes                   0% of     10               0% of    119
ALL                             71% of  8,072              73% of 133,554


## (a) Vector RAG over the article text

In [4]:
ANSWER_RULES = ("Return JSON {\"answers\": [...]} listing EVERY entity that answers the question, using the exact "
                "entity names; put the most likely answer first; do not include the topic entity itself unless it is "
                "an answer; return an empty list if you cannot find the answer.")
A_EMB = llm.embed("articles", [f"{a['title']}\n{a['text']}" for a in articles], cap=2000)
Q_EMB = llm.embed("questions", [q["question"] for q in Q])
K = 10
top = {q["qid"]: np.argsort(-(A_EMB @ Q_EMB[i]))[:K] for i, q in enumerate(Q)}
def body_json(msgs, tools=None, max_tokens=1500):
    b = LLM.body(msgs, max_tokens=max_tokens, tools=tools)
    b["response_format"] = {"type": "json_object"}
    return b
VEC = llm.run("vector", {q["qid"]: body_json([
    {"role": "system", "content": "Answer movie questions using ONLY the retrieved documents. " + ANSWER_RULES},
    {"role": "user", "content": f"Question: {q['question']}\n\nRetrieved documents:\n" +
     "\n".join(f"[{j + 1}] {articles[i]['title']}: {articles[i]['text']}" for j, i in enumerate(top[q['qid']]))}])
    for q in Q})
print(f"Topic film's own article in the top {K}: "
      f"{np.mean([key(q['topic']) in {key(articles[i]['movie']) for i in top[q['qid']]} for q in Q]):.0%}")

[vector] 900 new requests -> batch (0 cached)


[vector] resuming: 1 running, 0 queued


[vector] batch batch_6ad8307a220881908c1e2300f699e45a completed: 900 ok, 0 failed


[vector] 1 min: 0 running, 0 queued, 900 results cached


[vector] done in 0.5 min


Topic film's own article in the top 10: 100%


## (b) and (c) The notebook 8 graph agent over the curated KB and over the extracted graph

The tool, prompt and step budget (8 tool steps, then a final answer without tools) are notebook 8's. Both agents
run in lock-step: every call round advances all active questions of both systems by one step. Unlike the other
stages, these calls were sent synchronously (standard tier) rather than through the Batch API, to avoid hours of
Batch queueing per step.

In [5]:
# The agent stages run synchronously (standard tier, twice the Batch price): each of up to nine lock-step Batch jobs
# waited hours in the Batch queue, and the extra cost is about $1. Replays are unaffected (everything is cached).
if MODE == "batch":
    llm.mode = "test"
MAX_LIST, MAX_STEPS = 150, 8
class GraphTool:
    def __init__(self, G):
        self.G, self.lookup = G, {}
        for n in G.nodes():
            self.lookup.setdefault(key(n), n)
    def __call__(self, entities, relation=None):
        if isinstance(entities, str):
            entities = [entities]
        out, missing = defaultdict(set), 0
        for nm in entities:
            e = nm if nm in self.G else self.lookup.get(key(nm))
            if e is None:
                missing += 1; continue
            for _, t, r in self.G.out_edges(e, keys=True):
                if relation in (None, r): out[f"{r} (outgoing: the movie's {r.replace('_', ' ')} value)"].add(t)
            for s, _, r in self.G.in_edges(e, keys=True):
                if relation in (None, r): out[f"{r} (incoming: movies with this {r.replace('_', ' ')} value)"].add(s)
        result = {k: {"count": len(v), "entities": sorted(v)[:MAX_LIST]} for k, v in out.items()}
        if missing:
            result["not_found"] = missing
        if not out:
            result["note"] = "no edges with this relation for these entities; try another relation or omit it"
        return json.dumps(result, ensure_ascii=False)
TOOL = {"kb": GraphTool(KB), "extracted": GraphTool(EX)}
TOOLS = [{"type": "function", "function": {
    "name": "get_neighbors",
    "description": ("Traverse the movie knowledge graph. Movies are linked to their directors, writers, actors, release "
                    "year, language, genres and tags (relations: directed_by, written_by, starred_actors, release_year, "
                    "in_language, has_genre, has_tags, has_imdb_rating, has_imdb_votes). Returns neighbours in both "
                    "directions, labelled outgoing (movie -> value) or incoming (value -> movies). Pass several "
                    "entities at once to expand a whole set."),
    "parameters": {"type": "object", "properties": {
        "entities": {"type": "array", "items": {"type": "string"}},
        "relation": {"type": "string", "enum": sorted(RELS)}},
        "required": ["entities"]}}}]
AGENT_SYS = ("You answer movie questions by traversing a knowledge graph with the get_neighbors tool, one hop at a "
             "time, starting from the topic entity in [brackets]. " + ANSWER_RULES)

state = {f"{s}|{q['qid']}": {"msgs": [{"role": "system", "content": AGENT_SYS}, {"role": "user", "content": q["question"]}],
                             "answer": None, "calls": 0, "tokens": 0} for s in TOOL for q in Q}
for step in range(MAX_STEPS + 1):
    active = [k for k, s in state.items() if s["answer"] is None]
    if not active:
        break
    res = llm.run(f"agent_step{step + 1}", {k: body_json(state[k]["msgs"], tools=TOOLS if step < MAX_STEPS else None)
                                             for k in active})
    for k in active:
        r, s = res[k], state[k]
        s["tokens"] += r.get("in", 0) + r.get("out", 0)
        if not r.get("tool_calls"):
            s["answer"] = r.get("text") or "{}"; continue
        s["msgs"].append({"role": "assistant", "content": r["text"] or None,
                          "tool_calls": [{"id": t["id"], "type": "function", "function": {"name": t["name"], "arguments": t["arguments"]}}
                                         for t in r["tool_calls"]]})
        for t in r["tool_calls"]:
            s["calls"] += 1
            try:
                out = TOOL[k.split("|")[0]](**json.loads(t["arguments"]))
            except Exception as e:
                out = json.dumps({"error": f"{type(e).__name__}: {e}"})
            s["msgs"].append({"role": "tool", "tool_call_id": t["id"], "content": out})
    print(f"step {step + 1}: {len(active)} agents active")

[agent_step1] 1,800 new requests -> test (0 cached)


step 1: 1800 agents active
[agent_step2] 1,800 new requests -> test (0 cached)


step 2: 1800 agents active
[agent_step3] 1,256 new requests -> test (0 cached)


step 3: 1256 agents active
[agent_step4] 648 new requests -> test (0 cached)


step 4: 648 agents active
[agent_step5] 186 new requests -> test (0 cached)


step 5: 186 agents active
[agent_step6] 110 new requests -> test (0 cached)


step 6: 110 agents active
[agent_step7] 69 new requests -> test (0 cached)


step 7: 69 agents active
[agent_step8] 57 new requests -> test (0 cached)


step 8: 57 agents active
[agent_step9] 49 new requests -> test (0 cached)


step 9: 49 agents active


## Scoring

In [6]:
def parse_answers(text):
    try:
        return [str(a) for a in json.loads(text).get("answers", [])]
    except Exception:
        return []
def score(pred, gold):
    P, Gs = [key(p) for p in pred], {key(g) for g in gold}
    tp = len(set(P) & Gs)
    prec, rec = (tp / len(set(P)) if P else 0.0), tp / len(Gs)
    return {"hits1": bool(P) and P[0] in Gs, "f1": 2 * prec * rec / (prec + rec) if tp else 0.0}
SYS = {"vector": "(a) vector RAG", "kb": "(b) agent, curated KB", "extracted": "(c) agent, extracted graph"}
pred = {("vector", q["qid"]): parse_answers(VEC[q["qid"]]["text"]) for q in Q}
pred.update({(s, q["qid"]): parse_answers(state[f"{s}|{q['qid']}"]["answer"]) for s in TOOL for q in Q})
SC = {(s, q["qid"]): score(pred[(s, q["qid"])], q["gold"]) for s in SYS for q in Q}

def rel_path(q, maxn=20000):
    # The relation path that generates the gold set from the topic in the curated KB (MetaQA's template), or None.
    gold = {key(g) for g in q["gold"]}
    paths = [((), {q["topic"]})]
    for step in range(q["hop"]):
        nxt = []
        for p, F in paths:
            for r in RELS:
                for d in (("out",) if step % 2 == 0 else ("in",)):      # movie -> value -> movie -> value
                    N = set()
                    for e in F:
                        N |= {t for _, t, rr in KB.out_edges(e, keys=True) if rr == r} if d == "out" else \
                             {s for s, _, rr in KB.in_edges(e, keys=True) if rr == r}
                    if N and len(N) < maxn:
                        nxt.append((p + ((r, d),), N))
        paths = nxt
    for p, F in paths:
        if {key(x) for x in F} - {key(q["topic"])} == gold:
            return p
    return None
PATH = {q["qid"]: rel_path(q) for q in Q}
NOT_IN_TEXT = {"has_tags", "has_imdb_rating", "has_imdb_votes"}
def text_supported(q):
    p = PATH[q["qid"]]
    return p is not None and not any(r in NOT_IN_TEXT for r, _ in p)
def oracle_recall(q):
    # Follow the gold relation path in the extracted graph: share of gold answers reachable at all.
    p, F = PATH[q["qid"]], {key(q["topic"])}
    if p is None:
        return None
    lk = {key(n): n for n in EX.nodes()}
    F = {lk[f] for f in F if f in lk}
    for r, d in p:
        F = {t for e in F for _, t, rr in EX.out_edges(e, keys=True) if rr == r} if d == "out" else \
            {s for e in F for s, _, rr in EX.in_edges(e, keys=True) if rr == r}
    gold = {key(g) for g in q["gold"]}
    return len({key(x) for x in F} & gold) / len(gold)

RES = {}
for subset, keep in (("all", lambda q: True), ("text-supported", text_supported)):
    print(f"\n=== {subset} questions ===")
    print(f"{'hop':<6}{'n':>4}" + "".join(f"{SYS[s] + ' Hits@1':>36}" for s in SYS) + f"{'(c)-(a)':>24}{'(b)-(c)':>24}")
    for h in (1, 2, 3):
        qs = [q for q in Q if q["hop"] == h and keep(q)]
        if not qs:
            continue
        row = {}
        for s in SYS:
            v = [SC[(s, q["qid"])]["hits1"] for q in qs]
            row[s] = (np.mean(v), *wilson(sum(v), len(v)), np.mean([SC[(s, q["qid"])]["f1"] for q in qs]))
        hv = {s: [float(SC[(s, q["qid"])]["hits1"]) for q in qs] for s in SYS}
        d_ca, d_bc = paired_boot(hv["vector"], hv["extracted"]), paired_boot(hv["extracted"], hv["kb"])
        RES[(subset, h)] = (len(qs), row, d_ca, d_bc)
        fmt = lambda t: f"{t[0]:.0%} [{t[1]:.0%}-{t[2]:.0%}] F1 {t[3]:.2f}"
        fd = lambda d: f"{d[0] * 100:+.1f} [{d[1] * 100:+.1f}, {d[2] * 100:+.1f}]"
        print(f"{h}-hop{len(qs):>5}" + "".join(f"{fmt(row[s]):>36}" for s in SYS) + f"{fd(d_ca):>24}{fd(d_bc):>24}")
print("\nGold relation path found in the curated KB: " +
      ", ".join(f"{h}-hop {np.mean([PATH[q['qid']] is not None for q in Q if q['hop'] == h]):.0%}" for h in (1, 2, 3)))
print("Text-supported (path avoids tags and IMDb buckets, which the articles do not state): " +
      ", ".join(f"{h}-hop {np.mean([text_supported(q) for q in Q if q['hop'] == h]):.0%}" for h in (1, 2, 3)))
OR = {h: [x for q in Q if q["hop"] == h for x in [oracle_recall(q)] if x is not None] for h in (1, 2, 3)}
print("Gold answers reachable along the gold path in the extracted graph (oracle traversal): " +
      ", ".join(f"{h}-hop {np.mean(OR[h]):.0%}" for h in (1, 2, 3)))


=== all questions ===
hop      n               (a) vector RAG Hits@1        (b) agent, curated KB Hits@1   (c) agent, extracted graph Hits@1                 (c)-(a)                 (b)-(c)
1-hop  300               60% [55%-66%] F1 0.57               85% [81%-89%] F1 0.88               74% [68%-78%] F1 0.72     +13.3 [+8.0, +18.7]     +11.3 [+7.3, +15.7]
2-hop  300                 9% [6%-13%] F1 0.02             100% [99%-100%] F1 1.00               76% [71%-80%] F1 0.81    +67.0 [+61.0, +72.3]    +24.0 [+19.3, +29.0]
3-hop  300                12% [9%-17%] F1 0.06               93% [90%-95%] F1 0.92               69% [64%-74%] F1 0.67    +56.7 [+50.3, +62.7]    +24.0 [+18.7, +29.3]

=== text-supported questions ===
hop      n               (a) vector RAG Hits@1        (b) agent, curated KB Hits@1   (c) agent, extracted graph Hits@1                 (c)-(a)                 (b)-(c)
1-hop  240               70% [64%-76%] F1 0.67               96% [93%-98%] F1 0.96               85% [79%-89

Gold answers reachable along the gold path in the extracted graph (oracle traversal): 1-hop 85%, 2-hop 91%, 3-hop 79%


## Cost

In [7]:
LEDGER = llm.print_ledger()
for s in SYS:
    if s == "vector":
        continue
    st = [state[s + "|" + q["qid"]] for q in Q]
    print(f"{SYS[s]}: {np.mean([x['calls'] for x in st]):.1f} tool calls and "
          f"{np.mean([x['tokens'] for x in st]):,.0f} tokens per question")
print(f"(a) vector RAG: {np.mean([VEC[q['qid']]['in'] + VEC[q['qid']]['out'] for q in Q]):,.0f} tokens per question")

stage                           calls     input tok   output tok      USD  via Batch
agent_step1                     1,800       658,140       47,389     0.09         0%
agent_step2                     1,800       795,998       49,463     0.10         0%
agent_step3                     1,256       774,053      115,931     0.14         0%
agent_step4                       648       538,279       43,126     0.08         0%
agent_step5                       186       211,334       15,605     0.03         0%
agent_step6                       110       145,255        6,623     0.02         0%
agent_step7                        69       103,659        5,819     0.01         0%
agent_step8                        57       101,794        4,729     0.01         0%
agent_step9                        49        91,382        1,298     0.01         0%
extract                        18,127     8,771,301    1,934,382     0.92       100%
vector                            900     1,431,887       14,272 

In [8]:
print("=" * 100)
print(f"RESULTS SUMMARY — Notebook 19: MetaQA, {N_PER_HOP} questions per hop with a movie topic, gpt-6-luna")
print("=" * 100)
for h in (1, 2, 3):
    n, row, d_ca, d_bc = RES[("all", h)]
    print(f"{h}-hop n={n}: vector {row['vector'][0]:.0%} | curated KB {row['kb'][0]:.0%} | extracted {row['extracted'][0]:.0%}"
          f" | extracted - vector {d_ca[0] * 100:+.1f} [{d_ca[1] * 100:+.1f}, {d_ca[2] * 100:+.1f}]"
          f" | curated - extracted {d_bc[0] * 100:+.1f} [{d_bc[1] * 100:+.1f}, {d_bc[2] * 100:+.1f}]")
print(f"Triple recall of the extracted graph (topic films): {REC['topic films']['ALL'][0]:.0%}")
print("\nLecture message (Slides 36, 40-41): the same agent on the same questions; the difference between (b) and (c) is")
print("what curation buys, the difference between (c) and (a) is what an extracted graph buys over plain retrieval.")

RESULTS SUMMARY — Notebook 19: MetaQA, 300 questions per hop with a movie topic, gpt-6-luna
1-hop n=300: vector 60% | curated KB 85% | extracted 74% | extracted - vector +13.3 [+8.0, +18.7] | curated - extracted +11.3 [+7.3, +15.7]
2-hop n=300: vector 9% | curated KB 100% | extracted 76% | extracted - vector +67.0 [+61.0, +72.3] | curated - extracted +24.0 [+19.3, +29.0]
3-hop n=300: vector 12% | curated KB 93% | extracted 69% | extracted - vector +56.7 [+50.3, +62.7] | curated - extracted +24.0 [+18.7, +29.3]
Triple recall of the extracted graph (topic films): 71%

Lecture message (Slides 36, 40-41): the same agent on the same questions; the difference between (b) and (c) is
what curation buys, the difference between (c) and (a) is what an extracted graph buys over plain retrieval.
